# Mid-circuit measurement with the Qubex provider

This notebook shows how mid-circuit measurement compiles to Qubex pulses
and what timing guarantees the executor gives:

- a qubit can be **measured in the middle of a circuit and measured again**
  later — each `measure` maps to its own classical bit
- the readout window **blocks the measured qubit's drive channel**: later
  gates on that qubit (including two-qubit gates) start after the window
- **other qubits keep operating** during the readout window
- same-shot feedback (`if_test`) and mid-circuit `reset` are **rejected up
  front** with clear errors

The setup (offline `qubex.Experiment` from [qubex-config/](qubex-config/),
topology generated from the calibration data) is the same as in
[tutorial.ipynb](tutorial.ipynb) — see that notebook for the walkthrough.

Requirements:

```bash
pip install "qiskit-qubex-provider[qubex,plot] @ git+https://github.com/orangekame3/qiskit-qubex-provider.git"
```

In [1]:
from qiskit import QuantumCircuit, transpile
from qubex import Experiment

from qiskit_qubex_provider import (
    QubexProvider,
    build_device_topology,
    build_pulse_schedule_timeline_figure,
    extract_pulse_timeline,
)

In [2]:
topology = build_device_topology(
    calib_note_path="qubex-config/calibration/calib_note.json",
    params_dir="qubex-config/params",
    name="4Q-DEMO",
    device_id="4Q-DEMO",
)
exp = Experiment(
    system_id="4Q-DEMO-SYS",
    muxes=[0],
    config_dir="qubex-config/config",
    params_dir="qubex-config/params",
    calib_note_path="qubex-config/calibration/calib_note.json",
    calibration_valid_days=10000,
    mock_mode=True,
)
backend = QubexProvider.from_experiment(exp, device_topology=topology).get_backend()
exp.qubit_labels

[qubex.experiment.experiment_context] WARNING: Skew file not found: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/config/skew.yaml


date: 2026-06-12 20:21:39


python: 3.13.11


qubex: 1.5.0b4


env: /Users/orangekame3/.cache/uv/builds-v0/.tmpxGsMZO


...

chip: 16Q-DEMO (16-qubit demo chip)


qubits: ['Q00', 'Q01', 'Q02', 'Q03']


muxes: ['MUX0']


boxes: ['DEMO2', 'DEMO1']


['Q00', 'Q01', 'Q02', 'Q03']

## A circuit that measures `q0` twice

`q0` is measured mid-circuit into clbit 0, then participates in a CX and is
measured again into clbit 1. `q1` runs a gate of its own while `q0` is
being read out, and is measured into clbit 2. There is no feedback — the
measurement outcomes are simply collected per shot.

In [3]:
qc = QuantumCircuit(4, 3)
qc.h(0)
qc.measure(0, 0)  # mid-circuit measurement
qc.x(1)
qc.cx(0, 1)
qc.measure(0, 1)  # q0 measured again, into a different clbit
qc.measure(1, 2)
qc.draw()

┌───┐┌─┐     ┌─┐   
q_0: ┤ H ├┤M├──■──┤M├───
     ├───┤└╥┘┌─┴─┐└╥┘┌─┐
q_1: ┤ X ├─╫─┤ X ├─╫─┤M├
     └───┘ ║ └───┘ ║ └╥┘
q_2: ──────╫───────╫──╫─
           ║       ║  ║ 
q_3: ──────╫───────╫──╫─
           ║       ║  ║ 
c: 3/══════╩═══════╩══╩═
           0       1  2

In [4]:
scheduled = transpile(qc, backend, scheduling_method="alap", optimization_level=1)
schedule = backend.validate(scheduled)[0]  # build pulses, no execution

## Pulse-level view

Things to look for in the timeline:

- **`RQ00` has two readout pulses** — the mid-circuit window (≈48–560 ns)
  and the final one. `RQ01` has only the final readout.
- **`Q00` is silent during its readout window**; the echoed cross-resonance
  of the CX (the `FlatTop` pulses on `Q00-Q01` and the echo `Drag` pulses
  on `Q00`) starts exactly when the window ends at 560 ns.
- **`Q01`'s `Drag` pulse sits at ≈536 ns, inside `Q00`'s readout window** —
  measuring one qubit does not stall the others (ALAP placed it as late as
  possible; with ASAP it would run at *t* = 0).

In [5]:
build_pulse_schedule_timeline_figure(schedule, title="Mid-circuit measurement").show()

The same facts, numerically — the two readout windows on `RQ00` and the
in-window pulse on `Q01`:

In [6]:
schedule.plot(title="scheduling_method=mid-circuit (waveform detail)")

In [7]:
timeline = extract_pulse_timeline(schedule)
{channel: timeline[channel] for channel in ("RQ00", "Q01")}

{'RQ00': [{'kind': 'pulse',
   'name': 'FlatTop',
   'start_ns': 48.0,
   'duration_ns': 512.0},
  {'kind': 'pulse',
   'name': 'FlatTop',
   'start_ns': 1200.0,
   'duration_ns': 512.0}],
 'Q01': [{'kind': 'pulse',
   'name': 'Drag',
   'start_ns': 536.0,
   'duration_ns': 24.0},
  {'kind': 'pulse',
   'name': 'FlatTop',
   'start_ns': 560.0,
   'duration_ns': 272.0},
  {'kind': 'pulse',
   'name': 'FlatTop',
   'start_ns': 856.0,
   'duration_ns': 272.0},
  {'kind': 'pulse', 'name': 'Drag', 'start_ns': 1152.0, 'duration_ns': 16.0}]}

## How outcomes map to classical bits

On hardware, `backend.run(qc, shots=...)` executes the schedule with
`state_classification=True`. Each `measure` becomes a
`(qubit_label, capture_index)` pair — here `("Q00", 0)`, `("Q00", 1)`, and
`("Q01", 0)` — so repeated measurements of one qubit land in distinct
clbits. Counts follow Qiskit bit ordering (clbit 0 is the least significant
bit): a key like `"011"` means *first* `q0` measurement = 1, *second* = 1,
`q1` = 0.

## What is rejected

Same-shot feedback and mid-circuit reset cannot run on this pipeline, and
the executor refuses them at schedule-build time — before any hardware is
touched:

In [8]:
feedback = QuantumCircuit(1, 1)
feedback.measure(0, 0)
with feedback.if_test((feedback.clbits[0], True)):
    feedback.x(0)

try:
    backend.validate(feedback)
except ValueError as exc:
    print("if_test:", exc)

mid_reset = QuantumCircuit(1, 2)
mid_reset.x(0)
mid_reset.measure(0, 0)
mid_reset.reset(0)
mid_reset.measure(0, 1)

try:
    backend.validate(mid_reset)
except ValueError as exc:
    print("reset:", exc)

if_test: Classically controlled or dynamic Qiskit circuits are not supported by QubexPulseExecutor.
reset: Mid-circuit reset is not supported by QubexPulseExecutor.


## Wrap-up

Mid-circuit measurement works out of the box: measure, keep computing,
measure again — with the readout window enforced on the measured qubit's
drive channel and everything else free to proceed. For the timing model
details see
[docs/hardware-execution-notes.md](../docs/hardware-execution-notes.md);
for the full pipeline walkthrough see [tutorial.ipynb](tutorial.ipynb).